### Session 0 load depandancies and environmental variables

In [1]:
%reload_ext autoreload
%autoreload 2

In [ ]:
## a useful function to generate a data list for further analysis
import os,json,sys,pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import savemat
from scipy.stats import circmean
import seaborn as sns
from analyse_stimulus_evoked_response import main,plotting_visual_guided_response,plot_yaw_angular_velocity
##need to add this additional cell because useful tools are in another folder. Need to integrate these two folders one day
current_working_directory = Path.cwd()
parent_dir = current_working_directory.resolve().parents[0]
sys.path.insert(0, str(parent_dir) + "\\utilities")
from useful_tools import select_animals_gpt, find_file,get_fill_between_range,column_name_list,boolean_array_to_event_table,bootstrap_cumulative_probability
from data_cleaning import preprocess_fictrac_data,update_csv_value_pd
sys.path.insert(0, str(parent_dir) + "\\UnityDataAnalysis")
from trajectory_analysis import plot_scatter_violin

In [2]:
##introduce some helper class to define colour gradient
import matplotlib as mpl
from matplotlib import cm
class MplColorHelper:
    def __init__(self, cmap_name, start_val, stop_val):
        self.cmap_name = cmap_name
        self.cmap = plt.get_cmap(cmap_name)
        self.norm = mpl.colors.Normalize(vmin=start_val, vmax=stop_val)
        self.scalarMap = cm.ScalarMappable(norm=self.norm, cmap=self.cmap)

    def get_rgb(self, val):
        return self.scalarMap.to_rgba(val)
#colormap_name = "viridis"
colormap_name = "coolwarm"
# COL = MplColorHelper(colormap_name, 0, len(df["stim_type"].unique()))
# sm = cm.ScalarMappable(cmap=colormap_name)

#### Session 0.1: Load analysis methods in python dictionary form

In [3]:
#sheet_name="MatrexVR"
sheet_name="VCCball"
#sheet_name="Zball"
Datasets="Z:/DATA/experiment_trackball_Optomotor"
this_dataset_dir = f"{Datasets}/{sheet_name}"

In [4]:
json_file = "./analysis_methods_dictionary.json"
with open(json_file, "r") as f:
    analysis_methods = json.loads(f.read())

#### calculate autocorrelation

In [ ]:
## this cell is used to do batch processing of calculating autocorrelation of the stimulus sequence
from pathlib import Path
from calculate_autocorrelation import load_data
sheet_name="random_3duration_3coherence"
json_file = "./analysis_methods_dictionary.json"
with open(json_file, "r") as f:
    analysis_methods = json.loads(f.read())
Datasets=r"Z:\Users\chiyu\random_seed_autocorrelation_analysis"
this_dataset_dir = f"{Datasets}/{sheet_name}"
file_type=".csv"
for this_file in os.listdir(this_dataset_dir):
    if this_file.endswith(file_type):
        load_data(Path(this_dataset_dir) / this_file,analysis_methods)

In [ ]:
def batch_calculate_stim_bias(dir_list,partial_analysis):
    ISI_ID_lag=np.zeros((len(dir_list),5))
    stim_ID_lag=np.zeros((len(dir_list),5))
    stim_type_lag=np.zeros((len(dir_list),5))
    stim_direction_arr=np.zeros((len(dir_list),2))
    ISI_type_arr=np.zeros((len(dir_list),2))
    stim_type_arr=np.zeros((len(dir_list),7))
    i=0
    for this_dir in dir_list:    
        trial_pattern = "trial*.csv"
        csv_file_directory = find_file(this_dir, trial_pattern)
        stim_direction,ISI_type,stim_type,lag_data=load_data(csv_file_directory, json_file, partial_analysis)
        stim_type_arr[i,:]=stim_type
        stim_direction_arr[i,:]=stim_direction
        ISI_type_arr[i,:]=ISI_type
        ISI_ID_lag[i,:]=lag_data[0,:]
        stim_ID_lag[i,:]=lag_data[1,:]
        stim_type_lag[i,:]=lag_data[2,:]
        i=i+1
    print(f'{stim_type_arr.sum(axis=0)}')
    print(f'{stim_direction_arr.sum(axis=0)}')
    print(f'{ISI_type_arr.sum(axis=0)}')
    print(f'{ISI_ID_lag.mean(axis=0)}')
    print(f'{stim_ID_lag.mean(axis=0)}')
    print(f'{stim_type_lag.mean(axis=0)}')

In [ ]:
batch_calculate_stim_bias(dir_list,'early')
batch_calculate_stim_bias(dir_list,'middle')
batch_calculate_stim_bias(dir_list,'late')
'''
early
[424. 479. 540. 643. 589. 769. 532.]
424+532=956
479+769=1248
[1443. 1890.]
[1845. 2131.]
[-0.04762321 -0.1164732   0.01016095 -0.03843581 -0.14988475]
[ 0.00915247 -0.12262615 -0.09840216 -0.03593258  0.04220985]
[-0.11599431 -0.06198225 -0.13341417 -0.03512159  0.08460859]
max is either around 0
middle
[610. 404. 620. 654. 499. 651. 538.]
610+538=1148
404+651=1055
[1634. 1688.]
[2005. 1971.]
[ 0.00699606 -0.0320632  -0.05037562  0.13221885 -0.1800306 ]
[-0.0411484   0.00134714  0.05059404 -0.06485119  0.03194663]
[ 0.02552978 -0.12638919 -0.17651796 -0.11105817 -0.0898633 ]
max is either around 0.1 or -0.2
late
[461. 628. 704. 560. 394. 667. 562.]
461+562=1023
628+667=1295
[1793. 1623.]
[1719. 2257.]
[-0.08652557  0.04749003 -0.11511646 -0.00731669  0.02468436]
[-0.1190309   0.06575864 -0.07625521  0.02529781  0.06616186]
[-0.10959492 -0.0464329   0.02816368 -0.04218896  0.03156868]
max is either around 0.1 or -0.2
'''

#### Session 0.2: Load animal directory as a list

In [ ]:
## to load all directories with behavioural session including electrophysiology data
dir_list = []
# Datasets=[r"Y:\GN25051",r"Y:\GN25052",r"Y:\GN25053",r"Y:\GN25054",r"Y:\GN25055",r"Y:\GN25056",r"Y:\GN25057",r"Y:\GN25058",r"Y:\GN25059",r"Y:\GN25060",r"Y:\GN25061",
#           r"Y:\GN25062",r"Y:\GN25063",r"Y:\GN25064",r"Y:\GN25065",r"Y:\GN25066",r"Y:\GN25067",r"Y:\GN25068",r"Y:\GN25069",r"Y:\GN26008",r"Y:\GN26007"]
#Datasets=[r"Y:\GN26078",r"Y:\GN26079",r"Y:\GN26080",r"Y:\GN26081",r"Y:\GN26082",r"Y:\GN26083",r"Y:\GN26084"]
#Datasets=[r"Y:\GN26091",r"Y:\GN26092",r"Y:\GN26093",r"Y:\GN26094",r"Y:\GN26095",r"Y:\GN26096",r"Y:\GN26097",r"Y:\GN26098",r"Y:\GN26099",r"Y:\GN26100"]
Datasets=[r"Y:\GN26034",r"Y:\GN26035",r"Y:\GN26036",r"Y:\GN26037",r"Y:\GN26038",r"Y:\GN26039"]
#Datasets=[r"Y:\GN26059",r"Y:\GN26060",r"Y:\GN26061",r"Y:\GN26062",r"Y:\GN26063",r"Y:\GN26064",r"Y:\GN26065",r"Y:\GN26066",r"Y:\GN26067",r"Y:\GN26068",r"Y:\GN26069",r"Y:\GN26070",r"Y:\GN26071",r"Y:\GN26072",r"Y:\GN26073"]
for thisDataset in Datasets:
    for root, dirs, files in os.walk(thisDataset):
        for folder in dirs:
            if folder.startswith("Record"):
                dir_interest=Path(root)
                dir_list.append(dir_interest.parent)

In [ ]:
dir_list[1:7:2]

In [ ]:
dir_list[56:57]

In [ ]:
## to load one animal's all behavioural session in z ball (wihtout electrophysiology)
dir_list = []
file_type=".dat"
#Datasets=[r"Z:\DATA\experiment_trackball_Optomotor\Zball\GN26109",r"Z:\DATA\experiment_trackball_Optomotor\Zball\GN26108",r"Z:\DATA\experiment_trackball_Optomotor\Zball\GN26107",r"Z:\DATA\experiment_trackball_Optomotor\Zball\GN26106"]
Datasets=[r"Z:\DATA\experiment_trackball_Optomotor\Zball\GN26129",r"Z:\DATA\experiment_trackball_Optomotor\Zball\GN26130",r"Z:\DATA\experiment_trackball_Optomotor\Zball\GN26131",r"Z:\DATA\experiment_trackball_Optomotor\Zball\GN26132",
          r"Z:\DATA\experiment_trackball_Optomotor\Zball\GN26133",r"Z:\DATA\experiment_trackball_Optomotor\Zball\GN26134",r"Z:\DATA\experiment_trackball_Optomotor\Zball\GN26135",r"Z:\DATA\experiment_trackball_Optomotor\Zball\GN26136"]
for thisDataset in Datasets:
    for root, dirs, files in os.walk(thisDataset):
        for folder in dirs:
            if folder.startswith("session"):
                folder_path=os.path.join(root,folder)
                if any(name.endswith(file_type) for name in os.listdir(folder_path)):
                    dir_list.append(folder_path.replace("\\", "/"))

In [ ]:
## this cell is for sweeploom experiment. At that time, there was no subfolder for session
dir_list = []
file_type=".bonsai"
for root, dirs, files in os.walk(this_dataset_dir):
    if analysis_methods.get("experiment_name") in root.split(
        os.path.sep
    ):  ## add this condition to avoid data from other experiments
        folder_path=root
        if any(name.endswith(file_type) for name in os.listdir(folder_path)):
            dir_list.append(folder_path.replace("\\", "/"))
print(f"these directories are found {dir_list}")

In [ ]:
## this cell searches for a folder with a specified experiment_name under the dataset path and list up all the dat file in that folder.
## In this project, we usually have one dat file in that folder so there is no confusion
dir_list = []
file_type=".bonsai"
for root, dirs, files in os.walk(this_dataset_dir):
    if analysis_methods.get("experiment_name") in root.split(
        os.path.sep
    ):  ## add this condition to avoid data from other experiments
        for folder in dirs:
            if folder.startswith("session"):
                folder_path=os.path.join(root,folder)
                if any(name.endswith(file_type) for name in os.listdir(folder_path)):
                    dir_list.append(folder_path.replace("\\", "/"))
print(f"these directories are found {dir_list}")

In [ ]:
dir_list[3:12]

In [ ]:
dir_list2=dir_list[3:5]+dir_list[6:12]

In [ ]:
file_path=r"Z:\DATA\experiment_trackball_Optomotor\VCCball\sweeploom\linear_loom_1speed_3direction\sweeploom_linear_1speed_3_direction.pkl"
with open(file_path, 'rb') as file:
    dir_list = pickle.load(file)

In [ ]:
file_path=r"Z:\DATA\experiment_trackball_Optomotor\Zball\closed_loop_choices_6steps\dir_list_closed_loop_choices.pickle"
with open(file_path, 'rb') as file:
    test = pickle.load(file)

In [ ]:
with open('choices_open_loop_horizontal.pkl', 'wb') as f:
    pickle.dump(dir_list[-44:-22], f)

In [ ]:
with open('loom_horizontal.pkl', 'wb') as f:
    pickle.dump(final_list, f)

In [ ]:
dir_list_sweeping_conflict=dir_list[20:24:2]+dir_list[25:35:2]+dir_list[34:44:2]+dir_list[45:54:2]+dir_list[54::2]

In [ ]:
sweeping_rotation=dir_list[11:16:4]+dir_list[19:24:2]+dir_list[24:34:2]+dir_list[35:45:2]+dir_list[44:54:2]+dir_list[55::2]

In [ ]:
##if there is a need to update csv file, use this cell
for this_dir in dir_list:
    old_csv_pattern = "lux*.csv"
    old_csv_path = find_file(this_dir, old_csv_pattern)
    new_csv_pattern = "camera*.csv"
    new_csv_path = find_file(this_dir, new_csv_pattern)
    print(f"old_csv_path: {old_csv_path}")
    update_csv_value_pd(old_csv_path,new_csv_path,0)

### Session 1.0: Create curated dataset based on the list of directories

In [ ]:
pattern="database*.pickle"
for this_dir in dir_list:
    if any(Path(this_dir).glob(pattern)) and analysis_methods.get("overwrite_curated_dataset")==False:
        print(f"curated fictrac data found in {this_dir}. Skip this file")
        continue
    else:
        print(f"no curated fictrac data in {this_dir} or overwrite option is selected. Create curated file")
        preprocess_fictrac_data(this_dir,analysis_methods)

In [17]:
# Define the path to your Excel file
dir_list = []
# choose folders that contains fictrac .dat file
file_type=".dat"
using_google_sheet=True
experiment_name=analysis_methods.get("experiment_name")
if analysis_methods.get("load_experiment_condition_from_database") == True:
    if using_google_sheet==True:
        database_id = "1UL4eEUrQMapx9xz11-IyOSlPBcep1I9vBJ2uGgVudb8"
                #https://docs.google.com/spreadsheets/d/1UL4eEUrQMapx9xz11-IyOSlPBcep1I9vBJ2uGgVudb8/edit?usp=sharing
        url = f"https://docs.google.com/spreadsheets/d/{database_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"
        #df = pd.read_excel(url, engine='openpyxl')## use this function if the file is not google sheet but uploaded excel file

        df = pd.read_csv(url)
    else:
        excel_file_path = "Z:/DATA/experiment_trackball_Optomotor/Locusts Management.xlsx"
        print(f"using a database {excel_file_path} from the server but this file might be outdated")
        # Create a 'with' statement to open and read the Excel file
        with pd.ExcelFile(excel_file_path) as xls:
            # Read the Excel sheet into a DataFrame with the sheet name (folder name)
            df = pd.read_excel(xls, sheet_name)
        ##list up the conditions and answers as strings for input argument to select animal. One condition must pair with one answer
    if analysis_methods.get("select_animals_by_condition") == True:
        #animal_of_interest=select_animals_gpt(df,"Stimulus Type (list up all the stimulus paradigm this animal receive)","densities","Excluding this animal from analysis (Usually when animals die or molt, T/F)","F","Housing Condition before the experiment","solitarious")
        animal_of_interest=select_animals_gpt(df,"Independent variable1","coherence","Excluding this animal from analysis (Usually when animals die or molt, T/F)","F")
        #animal_of_interest=select_animals_gpt(df,"Independent variable1","conflict","Excluding this animal from analysis (Usually when animals die or molt, T/F)","F")
    else:
        animal_of_interest=df

    ID_array=animal_of_interest["ID"].values
    #print(f"these are the ID of the animals {ID_array}")
    dir_list = [
    root.replace("\\", "/")
    for root, dirs, files in os.walk(this_dataset_dir)
    if any(ID in root for ID in ID_array)
    and experiment_name in root.split(os.path.sep)
    and any(name.endswith(file_type) for name in files)
]
else:
    ## this cell searches for a folder with a specified experiment_name under the dataset path and list up all the hdf5 file in that folder.
    ## However,some changes need to be made once we do sleap or deeplabcut where there are more than one H5 file generated
    for root, dirs, files in os.walk(this_dataset_dir):
        if analysis_methods.get("experiment_name") in root.split(os.path.sep):## add this condition to avoid data from other experiments
            if any(name.endswith(file_type) for name in files):
                dir_list.append(root.replace("\\", "/"))

### Session 2.0: Analyse visually guided response

In [ ]:
# locust loom sessions in ephys
#dir_list=[r"Y:\GN25051\251101\looming\session2",r"Y:\GN25053\251108\looming\session2",r"Y:\GN25053\251108\looming\session6",r"Y:\GN25058\251123\looming\session3",r"Y:\GN25060\251130\looming\session1",r"Y:\GN25061\251205\looming\session3"]
# loom 6 type 4 speed in ephys
#dir_list=[r"Y:\GN25061\251205\looming\session4",r"Y:\GN25060\251130\looming\session4",r"Y:\GN25059\251129\looming\session1",r"Y:\GN25058\251123\looming\session2",r"Y:\GN25057\251122\looming\session2", r"Y:\GN25053\251108\looming\session1",r"Y:\GN25052\251102\looming\session1", r"Y:\GN25051\251101\looming\session1"]
# loom 6 type 4 speed in ephys dark background thisDir = r"Y:\GN25053\251108\looming\session7"# walking session, also showing jumping epochs
# loom 6 type 4 speed sequential in ephys
#dir_list=[r"Y:\GN25053\251108\looming\session5",r"Y:\GN25060\251130\looming\session2", r"Y:\GN25062\251206\looming\session2",r"Y:\GN25061\251205\looming\session2"]

In [ ]:
dir_list=['Y:/GN26055/260509/looming/session1','Y:/GN26056/260509/looming/session1','Y:/GN26066/260524/looming/session1',r'Y:/GN26068/260530/looming/session1',r'Y:\GN26062\260517\looming\session1',r'Y:\GN26060\260516\looming\session1']

In [ ]:
dirD=[r'D:\GN26098\260815\choices\session1',r'D:\GN26099\260816\choices\session1',r'D:\GN26100\260816\choices\session1']

In [ ]:
dir_list1=dir_list[:9:3]+dir_list[10:19:3]+dir_list[20:21]

In [ ]:
dir_list[1:7:2]

In [ ]:
dir_list[16:17]

In [ ]:
dir_list2=dir_list1+dirD+dir_list[1:7:2]+dir_list[6:14:4]+dir_list[16:17]

In [ ]:
# because the import does not update the new version of python.
# Need to restart kernel 
summary_list=[]
output1_across_exp=[]
output2_across_exp=[]
output3_across_exp=[]
output4_across_exp=[]
output5_across_exp=[]
for count,this_dir in enumerate(dir_list[8:-10]):
    # if "archive" in this_dir:
    #     print(f"skip archive folder for {this_dir}")
    #     continue
    # else:
    summary,speed,rotation,travel_distance_whole_session,heading_angle_discretised_all,_,_,_,_=main(this_dir,analysis_methods)
    summary['ID']=count
    summary_list.append(summary)
    output1_across_exp.append(speed)
    output2_across_exp.append(rotation)
    output3_across_exp.append(travel_distance_whole_session)
    output4_across_exp.append(this_dir)
    output5_across_exp.append(heading_angle_discretised_all)

#### Session 2.1: add meta data

In [20]:
df=pd.DataFrame(pd.concat(summary_list,axis=0, ignore_index=True))

In [21]:
plt.close("all")  # Closes all currently registered figures
plt.switch_backend("TkAgg")

In [22]:
#this cell is only useful if the video file exists
#If I ever want to add path and vid to every single row of the summary
from itertools import chain, repeat
#https://stackoverflow.com/questions/14878538/duplicate-elements-in-a-list
ll = list(chain.from_iterable(repeat(e, speed.shape[0]) for e in output4_across_exp))
camera_num=[]
video_pattern="camera*.mp4"
#final_list = list((lambda id: find_file(id,video_pattern).stem.split("_")[0], output4_across_exp))
for id in ll:
    found_result=find_file(id,video_pattern)
    camera_num.append(found_result.stem.split("_")[0])

In [23]:
#this cell is only useful if the number of trials is the same across animals
if analysis_methods.get("experiment_name") !='choices':
    new_data = {'Path': ll, 'VR_no': camera_num}
    df = df.assign(**new_data)
growth_condition='gregarious'
df.insert(0, 'growth_condition', np.repeat(growth_condition,summary_list[0].shape[0]*len(summary_list)).tolist())
#df.insert(0, 'ID', np.repeat(np.arange(len(output4_across_exp)),summary_list[0].shape[0]).tolist())

#### Session 2.2 a particular cell for the conflict experiment

In [ ]:
## this is for conflict experiment where we have a single stimuli going different direction
df_rotation=np.concatenate(output2_across_exp)
colormap = np.array(["k", "r", "b", "g", "y", "m", "c", "k"])
y_axis_lim=[-0.5,0.5]
premature_turn=0.8
walk_threshold=1
camera_fps = analysis_methods.get("camera_fps")
analysis_window=analysis_methods.get("analysis_window")
if 'LocustTexture1' in df.columns:
    groups = df.groupby(['stim_type','LocustTexture1','R1'])
    duration_arr=df['present_trial_duration'].unique()
else:
    groups = df.groupby(['stim_type','present_trial_duration'])
    duration_arr=np.sort(df['present_trial_duration'].unique())
num_indep_variables=groups.ngroups
init_frame=abs(analysis_window[0])*camera_fps
fig, axes = plt.subplots(
    nrows=num_indep_variables, ncols=1, figsize=(18, 84), tight_layout=True
)
plot_count=0
c_angle_dependant_list1=[]
c_angle_dependant_list2=[]
c_angle_dependant_list3=[]
cc_angle_dependant_list1=[]
cc_angle_dependant_list2=[]
cc_angle_dependant_list3=[]

for idx, this_stim in groups:
    this_response=df_rotation[this_stim[(this_stim['travel_distance_isi']>walk_threshold)&(abs(this_stim['opto_index_isi'])<premature_turn)].index,:]
    #this_response=df_rotation[this_stim[this_stim['travel_distance_isi']>2.5].index,:]
    #this_response=df_rotation[this_stim.index,:]
    time_points = np.arange(this_response.shape[1])
    time_points_b = np.tile(time_points, (this_response.shape[0], 1))
    if duration_arr.shape[0]>1:
        stim_duration=idx[1]
    else:
        stim_duration=duration_arr[0]
    axes[plot_count].plot(
        np.mean(time_points_b, axis=0),
        np.mean(this_response, axis=0),
        color='r',
        alpha=0.2,
    )
    axes[plot_count].axvline(
            x=init_frame,
            color="black",
            linestyle="--",
            label="Stimulus Starts",
        )
    axes[plot_count].axvline(
        x=init_frame + camera_fps * stim_duration,
        color="black",
        linestyle="--",
        label="Stimulus Ends",
    )
    axes[plot_count].set(
        yticks=[y_axis_lim[0], 0, y_axis_lim[1]],
        ylim=[y_axis_lim[0], y_axis_lim[1]],
        ylabel="Angular velocity (rad/s)",
        xlabel="Time (frames)",
        title=f'{idx}'
    )
    if idx[0].split("_")[0]=="cc" and idx[0].split("_")[1]=="back":
        print(f"stimulus type: {idx[0]}, angle decrease when time increases so need to reverse the array")
        tmp=this_response[:,init_frame:init_frame+camera_fps * stim_duration]
        angle_dependant_response=tmp[:,::-1]
    elif idx[0].split("_")[0]=="cc" and idx[0].split("_")[1]=="front":
        print(f"stimulus type: {idx[0]}, angle decrease when time increases and jump from -180 to 180 so translocate second half of the reversed array")
        tmp=this_response[:,init_frame:init_frame+camera_fps * stim_duration]
        tmp_rev=tmp[:,::-1]
        angle_dependant_response=np.hstack((tmp_rev[:,int(tmp.shape[1]/2):],tmp_rev[:,:int(tmp.shape[1]/2)]))
    elif idx[0].split("_")[0]=="c" and idx[0].split("_")[1]=="back":
        print(f"stimulus type: {idx[0]}, angle increase throughout time so no need to convert the array")
        angle_dependant_response=this_response[:,init_frame:init_frame+camera_fps * stim_duration]
    else:
        print(f"stimulus type: {idx[0]}, angle increase throughout time but jumps from 180 back to -180 so translocate second half of the array")
        tmp=this_response[:,init_frame:init_frame+camera_fps * stim_duration]
        angle_dependant_response=np.hstack((tmp[:,int(tmp.shape[1]/2):],tmp[:,:int(tmp.shape[1]/2)]))
    if 'R1' in df.columns:
        if idx[1]==1 and idx[0].split("_")[0]=="cc":
            print(f"list 1 comes from locust texture {idx[1]}")
            cc_angle_dependant_list1.append(angle_dependant_response)
        elif idx[1]==0 and idx[2]==0 and idx[0].split("_")[0]=="cc":
            print(f"list 2 comes from locust texture {idx[1]} and colour{idx[0]}")
            cc_angle_dependant_list2.append(angle_dependant_response)
        elif idx[1]==0 and idx[2]!=0 and idx[0].split("_")[0]=="cc":
            print(f"list 3 comes from locust texture {idx[1]} and colour{idx[0]}")
            cc_angle_dependant_list3.append(angle_dependant_response)
        elif idx[1]==1 and idx[0].split("_")[0]=="c":
            c_angle_dependant_list1.append(angle_dependant_response)
        elif idx[1]==0 and idx[2]==0 and idx[0].split("_")[0]=="c":
            c_angle_dependant_list2.append(angle_dependant_response)
        elif idx[1]==0 and idx[2]!=0 and idx[0].split("_")[0]=="c":
            c_angle_dependant_list3.append(angle_dependant_response)
    else:
        if idx[1]==duration_arr[0] and idx[0].split("_")[0]=="cc":
            cc_angle_dependant_list1.append(angle_dependant_response)
        elif idx[1]==duration_arr[1] and idx[0].split("_")[0]=="cc":
            cc_angle_dependant_list2.append(angle_dependant_response)
        elif idx[1]==duration_arr[2] and idx[0].split("_")[0]=="cc":
            cc_angle_dependant_list3.append(angle_dependant_response)
        elif idx[1]==duration_arr[0] and idx[0].split("_")[0]=="c":
            c_angle_dependant_list1.append(angle_dependant_response)
        elif idx[1]==duration_arr[1] and idx[0].split("_")[0]=="c":
            c_angle_dependant_list2.append(angle_dependant_response)
        elif idx[1]==duration_arr[2] and idx[0].split("_")[0]=="c":
            c_angle_dependant_list3.append(angle_dependant_response)
    plot_count=plot_count+1

In [ ]:
def adjust_unbalanced_arrays(arr1,arr2):
    if arr2.shape[0]>arr1.shape[0]:
        arr2 = arr2[np.random.choice(arr2.shape[0], size=arr1.shape[0])]
    elif arr1.shape[0]>arr2.shape[0]:
        arr1 = arr1[np.random.choice(arr1.shape[0], size=arr2.shape[0])]
    return arr1,arr2

randomly_adjust_unblanced_array=True
use_mean=True
colormap_name = "coolwarm"
sm = cm.ScalarMappable(cmap=colormap_name)
COL = MplColorHelper(colormap_name, 0, int(num_indep_variables/2))
y_axis_lim=[-0.5,0.5]
degree_sign = u'\N{DEGREE SIGN}'
fig2,((ax1,ax4,ax7),(ax2,ax5,ax8),(ax3,ax6,ax9)) = plt.subplots(
    nrows=3, ncols=3, figsize=(12, 15), tight_layout=True,
    sharex=True#,sharey='row'
)
cc_angle_dependant_arr1=np.vstack(cc_angle_dependant_list1)
cc_angle_dependant_arr2=np.vstack(cc_angle_dependant_list2)
cc_angle_dependant_arr3=np.vstack(cc_angle_dependant_list3)
c_angle_dependant_arr1=np.vstack(c_angle_dependant_list1)
c_angle_dependant_arr2=np.vstack(c_angle_dependant_list2)
c_angle_dependant_arr3=np.vstack(c_angle_dependant_list3)

if use_mean:
    cc_angle_value1=np.mean(cc_angle_dependant_arr1, axis=0)
    cc_angle_value2=np.mean(cc_angle_dependant_arr2, axis=0)
    cc_angle_value3=np.mean(cc_angle_dependant_arr3, axis=0)
    c_angle_value1=np.mean(c_angle_dependant_arr1, axis=0)
    c_angle_value2=np.mean(c_angle_dependant_arr2, axis=0)
    c_angle_value3=np.mean(c_angle_dependant_arr3, axis=0)
else:#use median
    cc_angle_value1=np.median(cc_angle_dependant_arr1, axis=0)
    cc_angle_value2=np.median(cc_angle_dependant_arr2, axis=0)
    cc_angle_value3=np.median(cc_angle_dependant_arr3, axis=0)
    c_angle_value1=np.median(c_angle_dependant_arr1, axis=0)
    c_angle_value2=np.median(c_angle_dependant_arr2, axis=0)
    c_angle_value3=np.median(c_angle_dependant_arr3, axis=0)
## quantify p and m system
if randomly_adjust_unblanced_array==True:
    c1,cc1=adjust_unbalanced_arrays(c_angle_dependant_arr1,cc_angle_dependant_arr1)
    c2,cc2=adjust_unbalanced_arrays(c_angle_dependant_arr2,cc_angle_dependant_arr2)
    c3,cc3=adjust_unbalanced_arrays(c_angle_dependant_arr3,cc_angle_dependant_arr3)
    if use_mean:
        p_values3=np.mean(c3+cc3, axis=0)/2
        p_values2=np.mean(c2+cc2, axis=0)/2
        p_values1=np.mean(c1+cc1, axis=0)/2
        m_values3=np.mean(c3-cc3, axis=0)/2
        m_values2=np.mean(c2-cc2, axis=0)/2
        m_values1=np.mean(c1-cc1, axis=0)/2
    else:
        p_values3=np.median(c3+cc3, axis=0)/2
        p_values2=np.median(c2+cc2, axis=0)/2
        p_values1=np.median(c1+cc1, axis=0)/2
        m_values3=np.median(c3-cc3, axis=0)/2
        m_values2=np.median(c2-cc2, axis=0)/2
        m_values1=np.median(c1-cc1, axis=0)/2
else:
    p_values3=(c_angle_value3+cc_angle_value3)/2
    p_values2=(c_angle_value2+cc_angle_value2)/2
    p_values1=(c_angle_value1+cc_angle_value1)/2
    m_values3=(c_angle_value3-cc_angle_value3)/2
    m_values2=(c_angle_value2-cc_angle_value2)/2
    m_values1=(c_angle_value1-cc_angle_value1)/2

ax1.plot(
    np.linspace(-180,180,cc_angle_dependant_arr1.shape[1]),
    cc_angle_value1,
    color=COL.get_rgb(0),
    alpha=0.6,
    label=f'45{degree_sign}/s'
)
ax1.plot(
    np.linspace(-180,180,c_angle_dependant_arr1.shape[1]),
    c_angle_value1,
    color=COL.get_rgb(6),
    alpha=0.6,
    label=f'45{degree_sign}/s'
)
ax2.plot(
    np.linspace(-180,180,cc_angle_dependant_arr2.shape[1]),
    cc_angle_value2,
    color=COL.get_rgb(1),
    alpha=0.6,
    label=f'30{degree_sign}/s'
)
ax2.plot(
    np.linspace(-180,180,c_angle_dependant_arr2.shape[1]),
    c_angle_value2,
    color=COL.get_rgb(5),
    alpha=0.6,
    label=f'30{degree_sign}/s'
)
ax3.plot(
    np.linspace(-180,180,cc_angle_dependant_arr3.shape[1]),
    cc_angle_value3,
    color=COL.get_rgb(2),
    alpha=0.6,
    label=f'14.4{degree_sign}/s'
)
ax3.plot(
    np.linspace(-180,180,c_angle_dependant_arr3.shape[1]),
    c_angle_value3,
    color=COL.get_rgb(4),
    alpha=0.6,
    label=f'14.4{degree_sign}/s'
)
ax1.set(
    yticks=[y_axis_lim[0],y_axis_lim[0]/2,0,y_axis_lim[1]/2,y_axis_lim[1]],
    xticks=[-180,-90,0,90,180],
    ylim=[y_axis_lim[0],y_axis_lim[1]],
    xlim=[-180,180],
    ylabel="Angular velocity (rad/s)",
    #xlabel=f"Stimulus Position ({degree_sign})"
    )
ax1.legend(loc="upper left")
ax2.set(
    yticks=[y_axis_lim[0],y_axis_lim[0]/2,0,y_axis_lim[1]/2,y_axis_lim[1]],
    xticks=[-180,-90,0,90,180],
    ylim=[y_axis_lim[0],y_axis_lim[1]],
    xlim=[-180,180],
    ylabel="Angular velocity (rad/s)",
    #xlabel=f"Stimulus Position ({degree_sign})"
    )
ax2.legend(loc="upper left")
ax3.set(
    yticks=[y_axis_lim[0],y_axis_lim[0]/2,0,y_axis_lim[1]/2,y_axis_lim[1]],
    xticks=[-180,-90,0,90,180],
    ylim=[y_axis_lim[0],y_axis_lim[1]],
    xlim=[-180,180],
    ylabel="Angular velocity (rad/s)",
    xlabel=f"Stimulus Position ({degree_sign})"
    )
ax3.legend(loc="upper left")
ax4.plot(
    np.linspace(-180,180,cc_angle_dependant_arr1.shape[1]),
    p_values1,
    color='k',
    alpha=0.7,
    label=f'45{degree_sign}/s'
)
ax5.plot(
    np.linspace(-180,180,cc_angle_dependant_arr2.shape[1]),
    p_values2,
    color='k',
    alpha=0.4,
    label=f'30{degree_sign}/s'
)
ax6.plot(
    np.linspace(-180,180,cc_angle_dependant_arr3.shape[1]),
    p_values3,
    color='k',
    alpha=0.1,
    label=f'14.4{degree_sign}/s'
)
ax4.set(
    yticks=[y_axis_lim[0]/2,0,y_axis_lim[1]/2],
    xticks=[-180,-90,0,90,180],
    ylim=[y_axis_lim[0]/2,y_axis_lim[1]/2],
    xlim=[-180,180],
    ylabel="P(Ψ)(rad/s)",
    #xlabel=f"Stimulus Position ({degree_sign})"
    )
ax4.legend(loc="upper left")
ax5.set(
    yticks=[y_axis_lim[0]/2,0,y_axis_lim[1]/2],
    xticks=[-180,-90,0,90,180],
    ylim=[y_axis_lim[0]/2,y_axis_lim[1]/2],
    xlim=[-180,180],
    ylabel="P(Ψ)(rad/s)",
    #xlabel=f"Stimulus Position ({degree_sign})"
    )
ax5.legend(loc="upper left")
ax6.set(
    yticks=[y_axis_lim[0]/2,0,y_axis_lim[1]/2],
    xticks=[-180,-90,0,90,180],
    ylim=[y_axis_lim[0]/2,y_axis_lim[1]/2],
    xlim=[-180,180],
    ylabel="P(Ψ)(rad/s)",
    xlabel=f"Stimulus Position ({degree_sign})"
    )
ax6.legend(loc="upper left")
ax7.plot(
    np.linspace(-180,180,cc_angle_dependant_arr1.shape[1]),
    m_values1,
    color='k',
    alpha=0.7,
    label=f'45{degree_sign}/s'
)
ax8.plot(
    np.linspace(-180,180,cc_angle_dependant_arr2.shape[1]),
    m_values2,
    color='k',
    alpha=0.4,
    label=f'30{degree_sign}/s'
)
ax9.plot(
    np.linspace(-180,180,cc_angle_dependant_arr3.shape[1]),
    m_values3,
    color='k',
    alpha=0.1,
    label=f'14.4{degree_sign}/s'
)
ax7.set(
    yticks=[y_axis_lim[0]/2,0,y_axis_lim[1]/2],
    xticks=[-180,-90,0,90,180],
    ylim=[y_axis_lim[0]/2,y_axis_lim[1]/2],
    xlim=[-180,180],
    ylabel="M(Ψ)*v(rad/s)",
    #xlabel=f"Stimulus Position ({degree_sign})"
    )
ax7.legend(loc="upper left")
ax8.set(
    yticks=[y_axis_lim[0]/2,0,y_axis_lim[1]/2],
    xticks=[-180,-90,0,90,180],
    ylim=[y_axis_lim[0]/2,y_axis_lim[1]/2],
    xlim=[-180,180],
    ylabel="M(Ψ)*v(rad/s)",
    #xlabel=f"Stimulus Position ({degree_sign})"
    )
ax8.legend(loc="upper left")
ax9.set(
    yticks=[y_axis_lim[0]/2,0,y_axis_lim[1]/2],
    xticks=[-180,-90,0,90,180],
    ylim=[y_axis_lim[0]/2,y_axis_lim[1]/2],
    xlim=[-180,180],
    ylabel="M(Ψ)*v(rad/s)",
    xlabel=f"Stimulus Position ({degree_sign})")
ax9.legend(loc="upper left")
file_name=f"angle_dependant_response.svg"
fig2.savefig(file_name)

In [24]:
#if you want to wrap other matrix into pandas dataframe
c_name_list=column_name_list(output2_across_exp[0].shape[1],"angular_velocity_f")
tmp=pd.DataFrame(np.vstack(output2_across_exp),columns=c_name_list)
df_rest_index=df.reset_index(drop=True)
tmp=tmp.reset_index(drop=True)
df_combined=pd.concat([df_rest_index, tmp], axis=1)

In [ ]:
active_trials_only=analysis_methods.get("active_trials_only")
analysis_window=analysis_methods.get("analysis_window")
stim_duration=analysis_methods.get("stim_duration")
camera_fps=analysis_methods.get("camera_fps")
running_trial_distance_threshold=1*abs(analysis_window[0]+0.5)# should be the time prior to looming stimuli but needs to exclude the time of stimulus presentation
output1_arr=np.vstack(output1_across_exp)
if active_trials_only:
    active_trials=df_combined[df_combined['travel_distance_isi']>1]
    active_trials_ratio=active_trials.shape[0]/df_combined.shape[0]
    print(f"this ratio of trial was excluded from analysis: {1-active_trials_ratio}")
    e_arr=np.repeat(active_trials['stim_type'].values,output1_arr.shape[1])
    v_arr=np.hstack(output1_arr[active_trials.index,:])
    t_arr=np.tile(np.linspace(analysis_window[0],analysis_window[1],output1_arr.shape[1]),active_trials.shape[0])
else:
    e_arr=np.repeat(df_combined['stim_type'].values,output1_arr.shape[1])
    t_arr=np.tile(np.linspace(analysis_window[0],analysis_window[1],output1_arr.shape[1]),output1_arr.shape[0])
    v_arr=np.hstack(output1_arr)
data=np.column_stack((np.transpose(e_arr),np.transpose(t_arr),np.transpose(v_arr)))
ready_to_plot=pd.DataFrame(data,columns=['trial_type', 'Time (s)', 'Speed (cm/s)'])

In [ ]:
plt.rcParams.update({'font.size': 16})
plt.rcParams['font.family'] = 'Arial'
#plt.set_cmap('cividis')
# Set the axis line width to 2
plt.rcParams['ytick.major.width'] = 1.5
plt.rcParams['xtick.major.width'] = 1.5
# plt.rcParams['xtick.labelsize'] = 12
# plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['axes.linewidth'] = 1.5
plt.rc('font', weight ='bold')
#plt.rc('xtick.major', size = 5, pad = 7)
plt.rc('xtick', labelsize = 16)
plt.rc('ytick', labelsize = 16)
if ready_to_plot['trial_type'].values[0]=='null' or ready_to_plot['trial_type'].values[0]=='omit'  or ready_to_plot['trial_type'].values[0]=='gregarious_receding':
    custom_colors = [
    "#3498db","#E0BC09"
    ]   
elif analysis_methods.get("experiment_name")!='sweeploom':
    custom_colors = [
    "#E0BC09","#3498db"
    ]
elif len(stim_duration)>1:
    custom_colors = [
    "#FF0000","#6DB26D","#3498db","#0000FF","#008000","#EE6247"
    ]
else:
    custom_colors = [
    "#0000FF","#000000","#FF0000"
    ]
sns.lineplot(x='Time (s)', y='Speed (cm/s)',hue="trial_type",
             data=ready_to_plot,estimator='median', palette=custom_colors,errorbar=('ci', 95),legend=True)
if analysis_methods.get("experiment_name")=='sweeploom':
    plt.axvline(x=0, color='black', linestyle='-', label='Loom appears')
    plt.axvline(x=1, color='black', linestyle='--', label='Looming onset')
    plt.axvline(x=2, color='gray', linestyle='--', label='Looming offset')
    plt.axvline(x=3, color='gray', linestyle='-', label='Loom disppears')
    if len(stim_duration)>1:
        plt.axvline(x=3, color='gray', linestyle='--', label='Slow Loom offset')
        plt.axvline(x=4, color='gray', linestyle='-', label='Slow Loom disppears')
    plt.xticks([0, 2, 4])
    plt.yticks([0,5])
else:
    plt.axvline(x=-0.5, color='black', linestyle='--', label='agent appear')
    plt.axvline(x=0, color='red', linestyle='-', label='Looming starts')
    plt.axvline(x=4, color='red', linestyle='--', label='Looming ends')
    plt.xticks([-2, 0, 2, 4, 6])
    plt.yticks([0,2,4])
    plt.ylim([0,4])
plt.savefig("speed_profile.svg", dpi=300, bbox_inches='tight')
plt.savefig("speed_profile.png", dpi=300, bbox_inches='tight')

In [ ]:
stop_trials=active_trials[active_trials['stop_index']>camera_fps*abs(analysis_window[0])]

In [ ]:
for this_stim_type in active_trials['stim_type'].unique():
    response_probability=stop_trials[stop_trials['stim_type']==this_stim_type].shape[0]/active_trials[active_trials['stim_type']==this_stim_type].shape[0]
    print(f"response probability for stimulus {this_stim_type} is {response_probability}")

In [ ]:
# Draw a nested violinplot and split the violins for easier comparison
print(stop_trials.shape[0])
sns.violinplot(data=stop_trials, x="stop_index", hue="stim_type",
               split=True,gap=.1, inner="point", fill=False,common_norm=True,density_norm='count',native_scale=True, cut=0)
plt.xticks([150,200,600,650],['-0.5', '0', '4','4.5'])
plt.xlabel('Time (s)')

In [ ]:
stim_type=df_combined["stim_type"].unique()
stim_type=np.sort(stim_type)
COL = MplColorHelper(colormap_name, 0, len(stim_type)-1)
plotting_visual_guided_response(np.vstack(output1_across_exp),stim_type,df_combined,analysis_methods,10,COL)
#plotting_visual_guided_response(np.vstack(output2_across_exp),stim_type,df_combined,analysis_methods,10,COL)

#### Session 2.3: save the database

In [16]:
data_dict = {name: col.values for name, col in df.items()}
savemat(r'Z:\DATA\experiment_trackball_Optomotor\MatrexVR\matrex_vr_coherence_database.mat', data_dict)

In [15]:
df_rotation=np.concatenate(output2_across_exp)
summary_file_name = Path(this_dataset_dir) /"angular_velocity_summary.npy"
np.save(summary_file_name,df_rotation)

In [ ]:
file_name="MatrexVR_4coherence_database.parquet.gzip"
if type(this_dataset_dir)==str:
    this_dataset_dir=Path(this_dataset_dir)
df=pd.read_parquet(this_dataset_dir / file_name)

In [ ]:
df_aba=df.groupby('ID').agg(travel_distance=('travel_distance','mean'),travel_distance_isi=('travel_distance_isi','mean'),ID=('ID','first'))
sns.histplot(data=df_aba, x="travel_distance_isi")
plt.show()

In [28]:
plt.figure(figsize=(12, 5))
plt.axvline(x=10,ymin=0,ymax=1,color='red',ls='--')
plt.hist(df['travel_distance'],bins=100)
# for i,this_coherence in enumerate([0,33,66,100]):
#     plt.figure(figsize=(12, 5))
#     trial_to_analyse=abs(df['stim_type'])==this_coherence
#     plt.hist(df['travel_distance'][df['stim_type']==0],bins=100)
plt.savefig("travel_distance_historgram.png")
#plt.show()

In [8]:
def converting_follow_velocity(row):
    if row['stim_type'] <0:
        row['stim_type'] *= -1    # Check condition on the first column
        row.loc['angular_velocity_f0':] *= -1
    return row

In [9]:
df_angular_speed=df.apply(converting_follow_velocity, axis=1)

In [10]:
analysis_window=analysis_methods.get("analysis_window")
stim_duration=analysis_methods.get("stim_duration")
camera_fps=analysis_methods.get("camera_fps")
graph_colour_code=analysis_methods.get("graph_colour_code")

In [11]:
arr=df_angular_speed.iloc[:,camera_fps*abs(analysis_window[0])+16:camera_fps*(abs(analysis_window[0])+stim_duration)+16].values
# target_speed=64*np.pi/180 
# hit_target_epochs=arr>target_speed

In [12]:
#parameter space to explore
coherence_to_analyse=100
sample_sizes=np.arange(8,101,7)
stimulus_speed=64*np.pi/180
target_speeds=[round(0.5*stimulus_speed,2),round(stimulus_speed,2),round(2*stimulus_speed,2)]

In [ ]:
## fit results into different distribution
ready_to_plot_AICc=np.ones((len(target_speeds), sample_sizes.shape[0],5 ))
ready_to_plot_BIC=np.ones((len(target_speeds), sample_sizes.shape[0], 5))
for i, this_target_speed in enumerate(target_speeds):
    hit_target_epochs=arr>this_target_speed
    for j, this_sample_size in enumerate(sample_sizes):
        trial_to_analyse=abs(df['stim_type'])==coherence_to_analyse
        this_data=hit_target_epochs[df[trial_to_analyse].index,]
        output_table,qualifying=boolean_array_to_event_table(this_data,timepoints=np.arange(2500)/100,stimulus_onset=0,consecutive_samples=this_sample_size)
        every = Fit_Everything(failures=output_table[output_table["event"]==1]["time"].values, right_censored=output_table[output_table["event"]==0]["time"].values,exclude=["Normal_2P","Gumbel_2P","Lognormal_2P","Lognormal_3P","Loglogistic_2P","Loglogistic_3P","Gamma_2P","Gamma_3P","Weibull_CR","Weibull_3P"],show_probability_plot=False, print_results=False,show_best_distribution_probability_plot=False,show_histogram_plot=False,show_PP_plot=False)  # Fit the Weibull_2P
        sorted_pd=every.results.sort_values('Distribution')
        print(f"consecutive samples: {this_sample_size}, speed threshold:{this_target_speed}")
        ready_to_plot_AICc[i,j,:]=sorted_pd['AICc'].values
        ready_to_plot_BIC[i,j,:]=sorted_pd['BIC'].values

In [17]:
from matplotlib.colors import ListedColormap

cmap = ListedColormap(
    ["blue","cyan","red","magenta", "purple"],
    name="custom_5"
)

In [ ]:
xlims=[(13000,17500),(9500,17000),(3000,13500)]
yticks=[8,50,99]
ix, iy, iz = np.indices(ready_to_plot_AICc.shape)
fig,axes = plt.subplots(nrows=3, ncols=1, figsize=(30, 20), tight_layout=True, sharey=False#sharey='row'
)
for i in range(ready_to_plot_AICc.shape[0]):
    axes[i].scatter(ready_to_plot_AICc[i,:,:],np.transpose(np.tile(sample_sizes,(ready_to_plot_AICc.shape[2],1))),s=40,c=iz[i,:,:],cmap=cmap,marker="^",alpha=1)
    axes[i].set_xlim(xlims[i])
    axes[i].set_xticks(xlims[i])
    axes[i].set_yticks(yticks)
    for label in axes[i].get_xticklabels():
        label.set_fontsize(20)
    for label in axes[i].get_yticklabels():
        label.set_fontsize(20)
plt.show()

In [13]:
plt.close("all")  # Closes all currently registered figures
plt.switch_backend("TkAgg")

In [ ]:
def plot_survival_analysis(failures, right_censored):
    exp = Fit_Exponential_1P(failures=failures, right_censored=right_censored, show_probability_plot=True, print_results=True,color='blue')  # Fit the Weibull_2P
    wbf = Fit_Weibull_2P(failures=failures,right_censored=right_censored,show_probability_plot=True, print_results=True,color='red')
    wbf = Fit_Weibull_2P(failures=failures,right_censored=right_censored,show_probability_plot=True, print_results=True,color='black')
    wbf3 = Fit_Weibull_Mixture(failures=failures, right_censored=right_censored, show_probability_plot=True, print_results=True,color='purple')
    wbf2 = Fit_Weibull_DS(failures=failures, right_censored=right_censored, show_probability_plot=True, print_results=True,color='magenta')  # Fit the Weibull_2P
    # Create the subplots and in each subplot we will plot the parametric distribution and obtain the Kaplan Meier fit.
# Note that the plot_type is being changed each time
    plt.figure(figsize=(12, 5))
    plt.subplot(131)
    #KaplanMeier(failures=failures, right_censored=right_censored, plot_type='SF', print_results=False, label='Kaplan-Meier')
    wbf.distribution.SF(label='Weibull function')
    plt.legend()
    plt.title('SF')
    plt.xlabel('sample number')
    plt.subplot(132)
    KaplanMeier(failures=failures, right_censored=right_censored, plot_type='CDF', print_results=False, label='Kaplan-Meier')
    #wbf.distribution.CDF(label='Parametric')
    plt.legend()
    plt.title('CDF')
    plt.subplot(133)
    KaplanMeier(failures=failures, right_censored=right_censored, plot_type='CHF', print_results=False, label='Kaplan-Meier')
    #wbf.distribution.CHF(label='Parametric')
    plt.legend()
    plt.title('CHF')
    #plt.subplots_adjust(left=0.07, right=0.95, top=0.92, wspace=0.25)  # format the plot layout
    #plt.show()

In [108]:
def output_beta_value(failures, right_censored):
    # exp = Fit_Exponential_1P(failures=failures, right_censored=right_censored, show_probability_plot=False, print_results=True)  # Fit the Weibull_2P
    # wbf = Fit_Weibull_2P(failures=failures,right_censored=right_censored,show_probability_plot=False, print_results=True)
    # wbf2 = Fit_Weibull_DS(failures=failures, right_censored=right_censored, show_probability_plot=False, print_results=True)
    wbf3 = Fit_Weibull_Mixture(failures=failures, right_censored=right_censored, show_probability_plot=False, print_results=True)
    exp = Fit_Exponential_1P(failures=failures, show_probability_plot=True, print_results=True,color='blue')  # Fit the Weibull_2P
    wbf = Fit_Weibull_2P(failures=failures,show_probability_plot=True, print_results=True,color='red')
    wbf2 = Fit_Weibull_DS(failures=failures,show_probability_plot=True, print_results=True,color='black')
    return exp, wbf, wbf2,wbf3

In [ ]:
colours=["y","c","m","k"]
use_single_value=True
plot_raw_data=False
if use_single_value:
    stimulus_speed=target_speeds[1]
    hit_target_epochs=arr>stimulus_speed
else:
    stimulus_speed=[target_speeds[0],target_speeds[2]]
    hit_target_lower=arr>stimulus_speed[0]
    hit_target_higher=arr<stimulus_speed[1]
    hit_target_epochs=hit_target_higher & hit_target_lower
stimulus_onset=0
camera_fps=100
timepoints = np.arange(arr.shape[1]) / camera_fps
post_times = timepoints[timepoints > stimulus_onset] - stimulus_onset

ready_to_plot_beta=np.ones((sample_sizes.shape[0],4))
ready_to_plot_beta[ready_to_plot_beta==1]=np.nan
ready_to_plot_beta_lower=np.ones((sample_sizes.shape[0],4))
ready_to_plot_beta_lower[ready_to_plot_beta_lower==1]=np.nan
ready_to_plot_beta_upper=np.ones((sample_sizes.shape[0],4))
ready_to_plot_beta_upper[ready_to_plot_beta_upper==1]=np.nan
ready_to_plot_bic=np.ones((sample_sizes.shape[0],4))
ready_to_plot_bic[ready_to_plot_bic==1]=np.nan
ready_to_plot_aic=np.ones((sample_sizes.shape[0],4))
ready_to_plot_aic[ready_to_plot_aic==1]=np.nan


if plot_raw_data:
    #fig, axes = plt.subplots(nrows=1, ncols=len(sample_sizes), figsize=(12, 5), tight_layout=True, sharex=True, sharey=True)
    fig, axes = plt.subplots(nrows=1, ncols=sample_sizes.shape[0], figsize=(12, 5), tight_layout=True, sharex=True, sharey=True)
for j, this_sample_size in enumerate(sample_sizes):
    #fig2, axes = plt.subplots(figsize=(5, 14), tight_layout=True)
    for i,this_coherence in enumerate([0,33,66,100]):
        # if this_coherence<67:
        #     continue
        trial_to_analyse=abs(df['stim_type'])==this_coherence
        this_data=hit_target_epochs[df[trial_to_analyse].index,]
        these_id=df[trial_to_analyse]["ID"].values
        these_trials=(df[trial_to_analyse]["stim_index"].values-1)/2
        these_coherences=abs(df[trial_to_analyse]["stim_type"]).values
        output_table,qualifying=boolean_array_to_event_table(this_data,timepoints=np.arange(2500)/100,stimulus_onset=stimulus_onset,consecutive_samples=this_sample_size)
        if plot_raw_data:
            probability, lower, upper = bootstrap_cumulative_probability(
            qualifying,
            n_resamples=99,
            confidence_level=0.95,
            seed=42,
            subject_ids=None,  # Use animal_ids here for multiple animals.
        )
            event_times = post_times[:qualifying.shape[1]]
            axes[j].step(
                event_times, probability,
                where="post", color=colours[i], label="Cumulative probability",
            )
            axes[j].fill_between(
                event_times, lower, upper,
                step="post", color=colours[i], alpha=0.25,
                label="95% pointwise bootstrap CI",
            )
            axes[j].set(
                xlabel="Time since stimulus onset (s)",
                ylabel="Probability of Hit (%)",
                xticks=[0,25],
                ylim=(0, 1),
                yticks=[0,0.5,1],
                yticklabels=['0','50','100'],
            )
        else:
            exp, wbf, wbf2,wbf3=output_beta_value(output_table[output_table["event"]==1]["time"].values, output_table[output_table["event"]==0]["time"].values)
            ready_to_plot_beta_lower[j,i]=wbf.results['Lower CI'][1]
            ready_to_plot_beta_upper[j,i]=wbf.results['Upper CI'][1]
            ready_to_plot_beta[j,i]=wbf.results['Point Estimate'][1]
            ready_to_plot_aic[j,i]=wbf.goodness_of_fit['Value'][1]
            ready_to_plot_bic[j,i]=wbf.goodness_of_fit['Value'][2]
            # plot_survival_analysis(output_table[output_table["event"]==1]["time"].values, output_table[output_table["event"]==0]["time"].values)
            # plt.show()
        #ax.legend()
        #output_table.insert(0,"ID",these_id)
    #output_table,qualifying=boolean_array_to_event_table(hit_target_epochs,timepoints=np.arange(2500)/100,stimulus_onset=0,consecutive_samples=50)
    #plt.title("Propability of hit",fontsize=18)
    #plt.legend()
    if plot_raw_data:
        for label in axes[j].get_xticklabels():
            label.set_fontsize(16)
        for label in axes[j].get_yticklabels():
            label.set_fontsize(16)
if plot_raw_data:
    if use_single_value:
        png_name=f'threshold{stimulus_speed}_sample_size_comparison.png'
        svg_name=f'threshold{stimulus_speed}_sample_size_comparison.svg'
    else:
        png_name=f'threshold{stimulus_speed[0]}_to{stimulus_speed[1]}_sample_size_comparison.png'
        svg_name=f'threshold{stimulus_speed[0]}_to{stimulus_speed[1]}_sample_size_comparison.svg'
    fig.savefig(png_name)
    fig.savefig(svg_name)
#plt.show()

In [ ]:
#plot a table filling values
result_summary=np.vstack((ready_to_plot_beta_lower[6,:],ready_to_plot_beta[6,:],ready_to_plot_beta_upper[6,:],ready_to_plot_aic[6,:],ready_to_plot_bic[6,:]))
row_labels=[r'$\kappa$ Lower CI',r'$\kappa$',r'$\kappa$ Upper CI','AIC','BIC']
#row_labels=[r'$\lambda$ Lower CI',r'$\lambda$',r'$\lambda$ Upper CI','AIC','BIC']
result_table=pd.DataFrame(result_summary, columns=['0%', '33%', '66%','100%'],index=row_labels)

In [ ]:
#plot a table filling values
fig, ax = plt.subplots()
ax.axis("off")
table_plot=pd.plotting.table(ax,result_table,loc="center", cellLoc="center")

In [ ]:
## extract the shape parameter from Weibull distribution
#parameter space to explore
use_single_value=False
coherence_to_analyse=100
sample_sizes=np.arange(8,101,7)
#sample_sizes=np.arange(8,51,7)
stimulus_speed=64*np.pi/180
target_speeds=[round(0.5*stimulus_speed,2),round(stimulus_speed,2),round(2*stimulus_speed,2)]
#target_speeds=[round(0.5*stimulus_speed,2),round(stimulus_speed,2)]
consider_censored_data=True
if use_single_value:
    ready_to_plot_beta=np.ones((len(target_speeds), sample_sizes.shape[0]))
    ready_to_plot_beta[ready_to_plot_beta==1]=np.nan
    ready_to_plot_beta_lower=np.ones((len(target_speeds), sample_sizes.shape[0]))
    ready_to_plot_beta_lower[ready_to_plot_beta_lower==1]=np.nan
    ready_to_plot_beta_upper=np.ones((len(target_speeds), sample_sizes.shape[0]))
    ready_to_plot_beta_upper[ready_to_plot_beta_upper==1]=np.nan
else:
    ready_to_plot_beta=np.ones((1, sample_sizes.shape[0]))
    ready_to_plot_beta[ready_to_plot_beta==1]=np.nan
    ready_to_plot_beta_lower=np.ones((1, sample_sizes.shape[0]))
    ready_to_plot_beta_lower[ready_to_plot_beta_lower==1]=np.nan
    ready_to_plot_beta_upper=np.ones((1, sample_sizes.shape[0]))
    ready_to_plot_beta_upper[ready_to_plot_beta_upper==1]=np.nan

if use_single_value:
    stimulus_speed=target_speeds[0]
    hit_target_epochs=arr>stimulus_speed
    for i, this_target_speed in enumerate(target_speeds):
        hit_target_epochs=arr>this_target_speed
        for j, this_sample_size in enumerate(sample_sizes):
            trial_to_analyse=abs(df['stim_type'])==coherence_to_analyse
            this_data=hit_target_epochs[df[trial_to_analyse].index,]
            output_table,qualifying=boolean_array_to_event_table(this_data,timepoints=np.arange(2500)/100,stimulus_onset=0,consecutive_samples=this_sample_size)
            if consider_censored_data:
                wbf = Fit_Weibull_DS(failures=output_table[output_table["event"]==1]["time"].values,right_censored=output_table[output_table["event"]==0]["time"].values,show_probability_plot=False, print_results=True)
            else:
                wbf = Fit_Weibull_2P(failures=output_table[output_table["event"]==1]["time"].values,show_probability_plot=False, print_results=True)
            ready_to_plot_beta[i,j]=wbf.beta
            ready_to_plot_beta_lower[i,j]=wbf.beta_lower
            ready_to_plot_beta_upper[i,j]==wbf.beta_upper
else:
    stimulus_speed=[target_speeds[0],target_speeds[2]]
    hit_target_lower=arr>stimulus_speed[0]
    hit_target_higher=arr<stimulus_speed[1]
    hit_target_epochs=hit_target_higher & hit_target_lower
    for j, this_sample_size in enumerate(sample_sizes):
        trial_to_analyse=abs(df['stim_type'])==coherence_to_analyse
        this_data=hit_target_epochs[df[trial_to_analyse].index,]
        output_table,qualifying=boolean_array_to_event_table(this_data,timepoints=np.arange(2500)/100,stimulus_onset=0,consecutive_samples=this_sample_size)
        if consider_censored_data:
            wbf = Fit_Weibull_DS(failures=output_table[output_table["event"]==1]["time"].values,right_censored=output_table[output_table["event"]==0]["time"].values,show_probability_plot=False, print_results=True)
        else:
            wbf = Fit_Weibull_2P(failures=output_table[output_table["event"]==1]["time"].values,show_probability_plot=False, print_results=True)
        ready_to_plot_beta[0,j]=wbf.beta
        ready_to_plot_beta_lower[0,j]=wbf.beta_lower
        ready_to_plot_beta_upper[0,j]==wbf.beta_upper

In [21]:
def heatmap(data, row_labels, col_labels, ax=None,
            cbar_kw=None, cbarlabel="", **kwargs):
    """
    Create a heatmap from a numpy array and two lists of labels.

    Parameters
    ----------
    data
        A 2D numpy array of shape (M, N).
    row_labels
        A list or array of length M with the labels for the rows.
    col_labels
        A list or array of length N with the labels for the columns.
    ax
        A `matplotlib.axes.Axes` instance to which the heatmap is plotted.  If
        not provided, use current Axes or create a new one.  Optional.
    cbar_kw
        A dictionary with arguments to `matplotlib.Figure.colorbar`.  Optional.
    cbarlabel
        The label for the colorbar.  Optional.
    **kwargs
        All other arguments are forwarded to `imshow`.
    """

    if ax is None:
        ax = plt.gca()

    if cbar_kw is None:
        cbar_kw = {}

    # Plot the heatmap
    im = ax.imshow(data,cmap='coolwarm',vmin=0,vmax=2,**kwargs)

    # Create colorbar
    cbar = ax.figure.colorbar(im, ax=ax, orientation="horizontal", location='top',aspect=80 ,**cbar_kw)
    cbar.ax.set_ylabel(cbarlabel,rotation=-90,va="bottom")
    

    # Show all ticks and label them with the respective list entries.
    ax.set_xticks(range(data.shape[1]), labels=col_labels,
                  rotation=-30, rotation_mode="xtick")
    ax.set_yticks(range(data.shape[0]), labels=row_labels)
    # Let the horizontal axes labeling appear on top.
    ax.tick_params(top=False, bottom=True,
                   labeltop=False, labelbottom=True)

    # Turn spines off and create white grid.
    ax.spines[:].set_visible(False)

    ax.set_xticks(np.arange(data.shape[1]+1)-.5, minor=True)
    ax.set_xlabel("step size (n consecutive frames)", fontsize=14)
    ax.set_yticks(np.arange(data.shape[0]+1)-.5, minor=True)
    ax.set_ylabel(r'threshold (rad s$^{-1}$)', fontsize=14)
    ax.grid(which="minor", color="w", linestyle='-', linewidth=3)
    ax.tick_params(which="minor", bottom=False, left=False)

    return im, cbar

In [22]:
def annotate_heatmap(im, data=None, valfmt="{x:.2f}",
                     textcolors=("black", "white"),
                     threshold=None, **textkw):
    """
    A function to annotate a heatmap.

    Parameters
    ----------
    im
        The AxesImage to be labeled.
    data
        Data used to annotate.  If None, the image's data is used.  Optional.
    valfmt
        The format of the annotations inside the heatmap.  This should either
        use the string format method, e.g. "$ {x:.2f}", or be a
        `matplotlib.ticker.Formatter`.  Optional.
    textcolors
        A pair of colors.  The first is used for values below a threshold,
        the second for those above.  Optional.
    threshold
        Value in data units according to which the colors from textcolors are
        applied.  If None (the default) uses the middle of the colormap as
        separation.  Optional.
    **kwargs
        All other arguments are forwarded to each call to `text` used to create
        the text labels.
    """

    if not isinstance(data, (list, np.ndarray)):
        data = im.get_array()

    # Normalize the threshold to the images color range.
    if threshold is not None:
        threshold = im.norm(threshold)
    else:
        threshold = im.norm(data.max())/2.

    # Set default alignment to center, but allow it to be
    # overwritten by textkw.
    kw = dict(horizontalalignment="center",
              verticalalignment="center")
    kw.update(textkw)

    # Get the formatter in case a string is supplied
    if isinstance(valfmt, str):
        valfmt = matplotlib.ticker.StrMethodFormatter(valfmt)

    # Loop over the data and create a `Text` for each "pixel".
    # Change the text's color depending on the data.
    texts = []
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            kw.update(color=textcolors[int(im.norm(data[i, j]) > threshold)])
            text = im.axes.text(j, i, valfmt(data[i, j], None), **kw)
            texts.append(text)

    return texts

In [26]:
im, cbar=heatmap(ready_to_plot_beta,['range'],sample_sizes.astype(str).tolist())
texts = annotate_heatmap(im, valfmt="{x:.2f}",fontsize=8)
plt.show()

In [80]:
plt.close("all")  # Closes all currently registered figures
plt.switch_backend("TkAgg")

In [29]:
def scatter_3d_array(data, distribution_name,target_speed_str,sample_size_str,marker_size=20, alpha=0.7,
                     color_by_nz=False, cmap="viridis"):
    """
    Combine the 2D slice plots of an (nx, ny, nz) array into one
    3D scatter plot.

    Coordinates
    -----------
    x : values stored in data
    y : index along the ny dimension
    z : index along the nz dimension

    Parameters
    ----------
    data : array-like
        Numeric array with shape (nx, ny, nz).
    marker_size : float
        Scatter-marker size.
    alpha : float
        Marker opacity.
    color_by_nx : bool
        If True, colour points by their index along the nx dimension.
    cmap : str
        Matplotlib colormap used when color_by_nx=True.

    Returns
    -------
    fig, ax
        Matplotlib figure and 3D axes.
    """
    data = np.asarray(data)

    if data.ndim != 3:
        raise ValueError("data must have shape (nx, ny, nz)")

    nx, ny, nz = data.shape

    # Coordinates matching data.ravel() in C order.
    ix, iy, iz = np.indices(data.shape)

    values = data.ravel()
    nx_indices = ix.ravel()
    ny_indices = iy.ravel()
    nz_indices = iz.ravel()

    # Exclude NaN and infinite values.
    valid = np.isfinite(values)

    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection="3d")

    scatter_kwargs = {
        "s": marker_size,
        "alpha": alpha,
        "depthshade": False
    }

    if color_by_nz:

        scatter = ax.scatter(
            values[valid],
            ny_indices[valid],
            nx_indices[valid],
            c=nz_indices[valid],
            cmap=cmap,
            **scatter_kwargs
        )

        cbar = fig.colorbar(scatter, ax=ax, pad=0.1)
        print(cbar.ax.get_ylim())
        yticks = np.linspace(*cbar.ax.get_ylim(), ready_to_plot_BIC.shape[2]+1)[:-1]
        yticks += (yticks[1] - yticks[0]) / 2
        cbar.set_label("Models", fontsize=14)
        cbar.set_ticks(yticks, labels=distribution_name)
        cbar.ax.tick_params(length=0) 
    else:
        ax.scatter(
            values[valid],
            ny_indices[valid],
            nx_indices[valid],
            **scatter_kwargs
        )
    #sample_sizes=np.arange(1,51,7)
    ax.set_xlabel("BIC value", fontsize=14)
    ax.set_ylabel("step size (n consecutive samples)", fontsize=14)
    #ax.set_yticklabels(["8","15","22","29","36","43","50"])
    ax.set_yticklabels(sample_size_str)
    ax.set_zlabel("threshold (Rad/s)", fontsize=14)
    #ax.set_zticklabels(["0.3","0.7","1.1"])
    ax.set_zticklabels(target_speed_str)

    ax.set_yticks(np.arange(ny))
    print(np.arange(ny))
    ax.set_zticks(np.arange(nx))

    fig.tight_layout()
    return fig, ax

In [ ]:
y_tick_label_list = sample_sizes.astype(str).tolist()
z_tick_label_list = list(map(str, target_speeds))
fig, ax = scatter_3d_array(
    ready_to_plot_BIC[:,:,:],
    distribution_name=sorted_pd['Distribution'].values,
    target_speed_str=z_tick_label_list,
    sample_size_str=y_tick_label_list,
    color_by_nz=True
)

plt.show()

In [30]:
# #file_name="matrex_vr_coherence_database_yaw_ori.pickle"
# file_name="matrex_vr_coherence_4hr_database_yaw_ori.pickle"
# df_combined.to_pickle(Path(this_dataset_dir)/file_name)
#path_name = r'Z:\DATA\experiment_trackball_Optomotor\Zball\sweeping_conflict_database.parquet.gzip'
#path_name = r'Z:\DATA\experiment_trackball_Optomotor\MatrexVR\MatrexVR_4coherence_database.parquet.gzip'
path_name = r'Z:\DATA\experiment_trackball_Optomotor\MatrexVR\MatrexVR_4coherence_database_rolling_median.parquet.gzip'
#path_name = r'Z:\DATA\experiment_trackball_Optomotor\Zball\closed_loop_choices_database.parquet.gzip'
df_combined.to_parquet(path_name,compression='gzip')

In [ ]:
'''
df.to_csv(file_name)#file will be big when dataset is big
df.to_pickle(file_name)#easy to save
df.to_feather(file_name)#needs pyarrow and column names needs to be string
path_name = r'Z:\DATA\experiment_trackball_Optomotor\ANALYSIS\Sercan et al\density_database.parquet.gzip'
finished.to_parquet(path_name,compression='gzip')
finished=pd.read_parquet(path_name)
df.to_hdf(file_name, key='solitarious', mode='w')#better used in dataset that has certain structure
'''

#### Session 2.X: Analyse animal's behaviour with multi-engines

In [ ]:
##this cell start the multi-engines. Make sure to run only once
import time
import ipyparallel as ipp
def show_clusters():
    clusters = ipp.ClusterManager().load_clusters() 
    print("{:15} {:^10} {}".format("cluster_id", "state", "cluster_file")) 
    for c in clusters:
        cd = clusters[c].to_dict()
        cluster_id = cd['cluster']['cluster_id']
        controller_state = cd['controller']['state']['state']
        cluster_file = getattr(clusters[c], '_trait_values')['cluster_file']
        print("{:15} {:^10} {}".format(cluster_id, controller_state, cluster_file))
    return cluster_id

cluster = ipp.Cluster(n=6)
await cluster.start_cluster()
cluster_neuropc=show_clusters()

In [ ]:
cluster_neuropc=show_clusters()

In [ ]:
##input cluster_id from previous cell
rc = ipp.Client(cluster_id=cluster_neuropc)

# Create a DirectView for parallel execution
dview = rc.direct_view()

# Define a function for parallel processing
def process_directory(this_dir, analysis_methods):
    from analyse_stimulus_evoked_response import main
    # Check if the H5 file (curated dataset) exists
    summary,speed,rotation,travel_distance_whole_session=main(this_dir,analysis_methods)
    return (summary,speed,rotation,travel_distance_whole_session)

# Define analysis_methods

# Use parallel execution to process directories
results = dview.map_sync(process_directory, dir_list, [analysis_methods] * len(dir_list))

# Initialize result lists
summary_list=[]
output1_across_exp=[]
output2_across_exp=[]
output3_across_exp=[]
output4_across_exp=[]

# Collect and organize results
for result in results:
    if result is not None:
        summary,speed,rotation = result
        summary_list.append(summary)
        output1_across_exp.append(speed)
        output2_across_exp.append(rotation)
        output3_across_exp.append(travel_distance_whole_session)
        output4_across_exp.append(this_dir)

# Now, you have the results collected in the respective lists

In [ ]:
rc.shutdown()

#### Session 2.5: Load existing dataset

In [ ]:
#database_pattern = "matrexVR_coherence_database.pickle"
database_pattern = "matrex_vr_coherence_database_yaw_ori.pickle"
load_existing_dataset=True
this_database = find_file(this_dataset_dir, database_pattern)
velocity_pattern="velocity*.npy"
jaw_vector_pattern="*jaw_rotation.npy"
summary_pattern="behavioural*.pickle"
pattern_list=[velocity_pattern,jaw_vector_pattern,summary_pattern]
print(f"loading this {this_database}")
if this_database is None or this_database.is_file()==False:
    print("no post-analysis database created. Either run analysis cell to get the list of animal's response or assemble post-analysis dataset from individual animals")
    if len(summary_list)>0:
    #if 'summary_list' in locals():
        df=pd.DataFrame(pd.concat(summary_list))
        print("use local list created from jupyter notebook")
    else:
        velocity_list=[]
        summary_list=[]
        rotation_list=[]
        for this_pattern in pattern_list:
            for this_dir in dir_list:
                this_file = find_file(this_dir, this_pattern)
                if this_file.suffix==".npy":
                    print({f"npy file is found. Let's assemble them"})
                    if this_pattern==velocity_pattern:
                        velocity_list.append(np.load(this_file))
                    elif this_pattern==jaw_vector_pattern:
                        rotation_list.append(np.load(this_file))
                elif this_file.suffix==".pickle":
                    print({f"load individual animal's summary"})
                    summary_list.append(pd.read_pickle(this_file))
                else:
                    print({f"no particular file created in this path: {this_dir}. Skip this animal"})
                

        df_velocity=np.concatenate(velocity_list)
        df_rotation=np.concatenate(rotation_list)
        df=pd.DataFrame(pd.concat(summary_list,axis=0, ignore_index=True))
elif load_existing_dataset==True:
    df=pd.read_pickle(this_database)
    df.rename(columns = {'Unnamed: 0':'trial_number'}, inplace = True)
    print("load pickle into pandas dataframe")
else:
    print("do not know what to do here")

In [ ]:
df_filtered = df[~df['ID'].isin([3,7,15,16,19,35,38,65])]
df_interest_no_lazy= df_filtered.apply(converting_follow_velocity, axis=1)
df_interest = df.apply(converting_follow_velocity, axis=1)

In [ ]:
df_interest_no_lazy= df_filtered.apply(converting_follow_velocity, axis=1)
df_interest = df.apply(converting_follow_velocity, axis=1)

In [ ]:
#check the temperature at the onset of every experiment and plot the distribution
temperature_list=[]
num_trials=168
for i in range(int(df.shape[0]/num_trials)):
    temperature_list.append(df.iloc[i*num_trials]["temperature"])
plt.hist(temperature_list,bins=10)
plt.show()
#df_below35=df[df["ID"].isin(np.where(np.array(temperature_list)<35)[0])]
#df_above=df[df["ID"].isin(np.where(np.array(temperature_list)>35)[0])]

#### Session 3: plotting data

In [54]:
def plot_preview_summary(df,this_dataset_dir,analysis_methods,vr_no="all"):
    walk_threshold=200
    walk_threshold_tbt=0.5
    save_output= analysis_methods.get("save_output")
    visual_paradigm_name= analysis_methods.get("experiment_name")
    colormap = np.array(analysis_methods.get("graph_colour_code"))
    premature_turn=0.8#set this criteria to remove trials where animals starting to turn prior to the stimulus presentation
    fig, axes = plt.subplots(
        nrows=2, ncols=2, figsize=(18, 14), tight_layout=True
    )
    ax1, ax2, ax3, ax4 = axes.flatten()
    num_bins=10
    n, bins, patches = ax1.hist(df.groupby(['ID'])["travel_distance_isi"].sum().values,num_bins,color=colormap[3])
    ax1.set(
    ylabel="Count",
    xlabel="Walk distance (cm)")

    for id, entries in df.groupby(['ID']):
        if entries["travel_distance_isi"].sum()>walk_threshold:
            above_threshold=True
        else:
            above_threshold=False
        entries=entries[abs(entries["opto_index_isi"])<premature_turn]
        for idx, this_stim in entries.groupby("stim_type"):
            this_response=this_stim["opto_index"].values
            mean_response = np.mean(this_response, axis=0)
            sem_response = np.std(this_response, axis=0, ddof=1) / np.sqrt(
                this_response.shape[0]
            )
            if above_threshold == True:
                this_colour=colormap[5]#blue
            else:
                this_colour=colormap[6]
            ax2.errorbar(
                idx,
                mean_response,
                yerr=sem_response,
                c=this_colour,
                fmt="o",
                elinewidth=2,
                capsize=3,
            )
    ax2.set_ylim(-1, 1)

    if visual_paradigm_name =="gratings":
        xlabel="Temporal frequency (Hz)"
    elif visual_paradigm_name =="coherence":
        xlabel="Coherence (%)"
    elif visual_paradigm_name in ["sweeping","choices","conflict"]:
        xlabel="type"
    ax2.set(
        yticks=[-1,0,1],
        ylabel="Optomotor Index",
        xlabel=xlabel)
        
    num_bins=200
    #n, bins, patches = ax3.hist(df["travel_distance_isi"].values,num_bins,density=False,range=(0,10),color=colormap[3])
    n, bins, patches = ax3.hist(df["travel_distance"].values,num_bins,density=False,range=(0,10),color=colormap[3])
    ax3.set(
    ylabel="Density",
    xlabel="Walk distance (cm)",
    xticks=[0.5,1,2],
    )
    for this_coherence,entries in df.groupby("stim_type"):
        #this_response = entries[(entries["travel_distance_isi"]>walk_threshold_tbt)&(abs(entries["opto_index_isi"])<premature_turn)]["opto_index"].values
        #this_response = entries[(abs(entries["opto_index_isi"])<premature_turn)]["opto_index"].values
        this_response = entries[entries['stim_index']<16]["opto_index"].values
        #this_response = entries["opto_index"].values
        #this_response = entries[entries["travel_distance_isi"]>walk_threshold_tbt]["opto_index"].values
        mean_response = np.mean(this_response, axis=0)
        sem_response = np.std(this_response, axis=0, ddof=1) / np.sqrt(this_response.shape[0])
        ax4.errorbar(
            this_coherence,
            mean_response,
            yerr=sem_response,
            c=colormap[0],
            fmt="o",
            elinewidth=2,
            capsize=3,
        )
        #plot_scatter_violin(this_response,this_coherence)
    ax4.set_ylim(-1, 1)
    ax4.set(
        yticks=[-1,-0.5,0,0.5,1],
        ylabel="Optomotor Index",
        xlabel=xlabel)
    fig.suptitle(f'The response of locusts during {visual_paradigm_name} stimulus presentation')
    ax1.title.set_text('The total walk distance of every locust')
    ax2.title.set_text('The optomotor index of every locust')
    ax3.title.set_text('The walk distance of every trial')
    ax4.title.set_text('The optomotor index of every walk trial')
    fig_name=f"{vr_no}_bonfic_{visual_paradigm_name}_summary.svg"
    if save_output==True:
        fig.savefig(Path(this_dataset_dir)/fig_name)

In [55]:
plot_preview_summary(df,this_dataset_dir,analysis_methods)
plt.show()

In [ ]:
if analysis_methods.get("create_null_null_condition",False):
    stim_types=['black_black','glocust_glocust','black_glocust','glocust_black','black_null', 'glocust_null', 'null_black', 'null_glocust','null_null']
else:
    stim_types=['black_black','black_glocust','black_null','glocust_black','glocust_glocust','glocust_null','null_black','null_glocust']
# with open('list.pickle', 'wb') as file:
#     pickle.dump(output5_across_exp, file)
# with open('list.pickle', 'rb') as file:
#     output5_across_exp = pickle.load(file)
combinations = [
    [angle for animal in output5_across_exp for angle in animal[i]]
    for i in range(len(stim_types))
]

fig, axes = plt.subplots(1, len(stim_types), figsize=(24, 18),
                         subplot_kw={'projection': 'polar'})

for ax, data, title in zip(axes, combinations, stim_types):
    print(len(data))
    ax.hist(data, bins=12)
    #ax.set(title=title,yticks=[4])
    ax.plot(circmean(data),7,marker='*',color='r',label='circular mean')
    #ax.set(title=title,ylim=[0,8],yticks=[4],yticklabels=[''])
    ax.set_title(title)
plot_name='head_direction_across_animals.svg'
fig.savefig(plot_name)
# plt.tight_layout()
# plt.show()

In [ ]:
#define the first and last frames to analyse
stim_duration=analysis_methods.get("stim_duration")
interval_duration=analysis_methods.get("interval_duration")
if type(interval_duration) == list and type(stim_duration) == list:
    analysis_window = [
        min(interval_duration) * -1,
        max(stim_duration) + min(interval_duration),
    ]
elif type(interval_duration) == list:
    analysis_window = [
        min(interval_duration) * -1,
        stim_duration + min(interval_duration),
    ]
elif type(stim_duration) == list:
    analysis_window = [
        interval_duration * -1,
        max(stim_duration) + interval_duration,
    ]
else:
    analysis_window = [
        interval_duration * -1,
        stim_duration + interval_duration,
    ]
init_frame=analysis_methods.get("camera_fps")*abs(analysis_window[0])
end_frame=analysis_methods.get("camera_fps")*analysis_window[1]
print(init_frame,end_frame)

In [70]:
analysis_methods.update({'active_trials_only': False})
fig, fig_tbt=plot_yaw_angular_velocity(df_combined,analysis_methods,init_frame,end_frame,"all",colormap_name)

In [ ]:
vr_groups = df.groupby('VR_no')
for vr_no, entries in vr_groups:
    plot_preview_summary(entries,this_dataset_dir,analysis_methods,vr_no)